# Project 2 — EEG Motor Imagery Classification

Foundation project for the multimodal-intent-decoding repo (pairs with [Project 3: EEG+EMG fusion](02_eeg_emg_fusion.ipynb)).

**Dataset**: BCI Competition IV dataset 2a, accessed through **MOABB** (Mother of All BCI Benchmarks) — 9 subjects, 22 channels, 4 classes (left hand, right hand, feet, tongue), 250 Hz, 2 sessions/subject. MOABB handles fetching, applies a standardized evaluation protocol, and gives published leaderboard numbers to benchmark against — reporting a number in a vacuum is what separates a tutorial from this project.

Alternative for scale: PhysioNet EEG Motor Movement/Imagery (109 subjects, 64 channels, no data-use agreement).

## Pipeline
1. **MNE-Python**: bandpass 8–30 Hz, epoch 0.5–2.5 s post-cue, ICA against EOG channels for artifact removal.
2. **Baselines, escalating**: CSP + LDA → FBCSP (4 Hz sub-bands from 4–40 Hz, mutual-information feature selection) → Riemannian tangent-space projection + logistic regression (`pyriemann`). The Riemannian pipeline usually wins and is cheap — worth discovering yourself.
3. **Deep models** via `braindecode`: EEGNet and ShallowFBCSPNet.
4. **Evaluate**: within-session, cross-session (train session T, test session E), cross-subject.
5. **The differentiator**: add a rest/no-control class and report false activations per minute at varying confidence thresholds, not just 4-class accuracy. For any device that moves, a spurious command is a safety event — accuracy alone hides it.

Expected: within-subject 70–80%, cross-subject 50–65%. Chance is 25%, so say so.


## 1. Setup

In [ ]:
# Colab setup
# !pip install -q moabb mne braindecode pyriemann scikit-learn torch numpy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne
from mne.decoding import CSP

import moabb
from moabb.datasets import BNCI2014_001  # BCI Competition IV 2a, MOABB's canonical name
from moabb.paradigms import MotorImagery

from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

try:
    from pyriemann.estimation import Covariances
    from pyriemann.tangentspace import TangentSpace
    HAS_PYRIEMANN = True
except ImportError:
    HAS_PYRIEMANN = False
    print("pyriemann not installed — pip install pyriemann to enable the Riemannian arm")

try:
    import torch
    from braindecode.models import EEGNet, ShallowFBCSPNet
    from braindecode.classifier import EEGClassifier
    from skorch.callbacks import EarlyStopping
    HAS_BRAINDECODE = True
except ImportError:
    HAS_BRAINDECODE = False
    print("braindecode/skorch not installed — pip install braindecode skorch for the deep-model arms")

mne.set_log_level("WARNING")
moabb.set_log_level("WARNING")
RNG_SEED = 0
np.random.seed(RNG_SEED)


## 2. Config

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    fmin: float = 8.0
    fmax: float = 30.0
    tmin: float = 0.5
    tmax: float = 2.5
    fbcsp_bands: tuple = tuple((lo, lo + 4) for lo in range(4, 40, 4))  # 4-40 Hz, 4 Hz sub-bands
    n_csp_components: int = 6
    resting_label: str = "rest"

CFG = Config()


## 3. Load via MOABB + MNE preprocessing

In [ ]:
dataset = BNCI2014_001()
paradigm = MotorImagery(fmin=CFG.fmin, fmax=CFG.fmax, tmin=CFG.tmin, tmax=CFG.tmax)

subjects = dataset.subject_list
print(f"{len(subjects)} subjects: {subjects}")


def load_subject_epochs(subject_id):
    """MOABB returns (X, y, metadata) — X is already band-passed per `paradigm`,
    metadata carries subject/session/run for the split protocols below."""
    X, y, metadata = paradigm.get_data(dataset=dataset, subjects=[subject_id])
    return X, y, metadata


def run_ica_artifact_removal(raw: mne.io.Raw, eog_channels=("EOG1", "EOG2", "EOG3")):
    """For datasets exposing raw continuous data (not needed for MOABB's pre-epoched
    X/y/metadata path above, but included for completeness / if you load raw .gdf directly)."""
    ica = mne.preprocessing.ICA(n_components=15, random_state=RNG_SEED, max_iter="auto")
    ica.fit(raw)
    present_eog = [ch for ch in eog_channels if ch in raw.ch_names]
    if present_eog:
        eog_indices, _ = ica.find_bads_eog(raw, ch_name=present_eog)
        ica.exclude = eog_indices
    return ica.apply(raw.copy())


## 4. Baseline 1: CSP + LDA

In [ ]:
def csp_lda_pipeline(n_components=CFG.n_csp_components):
    return Pipeline([
        ("csp", CSP(n_components=n_components, reg=None, log=True, norm_trace=False)),
        ("lda", LinearDiscriminantAnalysis()),
    ])


## 5. Baseline 2: FBCSP (filter-bank CSP + mutual-info feature selection)

In [ ]:
def bandpass_epochs(X, sfreq, lo, hi, order=4):
    from scipy.signal import butter, sosfiltfilt
    sos = butter(order, [lo, hi], btype="bandpass", fs=sfreq, output="sos")
    return sosfiltfilt(sos, X, axis=-1)


def fbcsp_features(X, y, sfreq, bands, n_components=4, fit_csps=None):
    """Band-pass into each sub-band, fit (or apply) CSP per band, concat log-variance
    features, return the feature matrix + the fitted per-band CSP list."""
    feats = []
    csps = fit_csps if fit_csps is not None else []
    fitting = fit_csps is None
    for i, (lo, hi) in enumerate(bands):
        Xb = bandpass_epochs(X, sfreq, lo, hi)
        if fitting:
            csp = CSP(n_components=n_components, reg="ledoit_wolf", log=True, norm_trace=False)
            csp.fit(Xb, y)
            csps.append(csp)
        else:
            csp = csps[i]
        feats.append(csp.transform(Xb))
    return np.concatenate(feats, axis=1), csps


class FBCSPClassifier:
    """MI-based band selection on top of concatenated per-band CSP log-variance features."""
    def __init__(self, sfreq, bands=CFG.fbcsp_bands, n_components=4, k_best=20):
        self.sfreq, self.bands, self.n_components, self.k_best = sfreq, bands, n_components, k_best
        self.csps = None
        self.selector = SelectKBest(mutual_info_classif, k=k_best)
        self.clf = LinearDiscriminantAnalysis()

    def fit(self, X, y):
        F, self.csps = fbcsp_features(X, y, self.sfreq, self.bands, self.n_components)
        Fs = self.selector.fit_transform(F, y)
        self.clf.fit(Fs, y)
        return self

    def predict(self, X):
        F, _ = fbcsp_features(X, None, self.sfreq, self.bands, self.n_components, fit_csps=self.csps)
        Fs = self.selector.transform(F)
        return self.clf.predict(Fs)


## 6. Baseline 3: Riemannian tangent space + logistic regression

In [ ]:
def riemannian_pipeline():
    if not HAS_PYRIEMANN:
        raise RuntimeError("pyriemann not installed")
    return Pipeline([
        ("cov", Covariances(estimator="oas")),
        ("ts", TangentSpace(metric="riemann")),
        ("logreg", LogisticRegression(max_iter=1000)),
    ])


## 7. Deep models: EEGNet and ShallowFBCSPNet (braindecode)

In [ ]:
def make_eegnet(n_channels, n_classes, n_times):
    return EEGNet(n_chans=n_channels, n_outputs=n_classes, n_times=n_times)


def make_shallow_fbcsp(n_channels, n_classes, n_times):
    return ShallowFBCSPNet(n_chans=n_channels, n_outputs=n_classes, n_times=n_times, final_conv_length="auto")


def make_eeg_classifier(module, max_epochs=100, lr=1e-3):
    from skorch.dataset import ValidSplit
    return EEGClassifier(
        module, criterion=torch.nn.CrossEntropyLoss, optimizer=torch.optim.Adam,
        optimizer__lr=lr, train_split=ValidSplit(cv=0.2, stratified=True, random_state=RNG_SEED),
        max_epochs=max_epochs, batch_size=32,
        callbacks=[EarlyStopping(patience=15, monitor='valid_loss')], verbose=0,
    )


## 8. Split protocols: within-session, cross-session, cross-subject

In [ ]:
def within_session_split(metadata, session_col="session"):
    """70/30 split inside each session, stratified — returns boolean train/test masks."""
    rng = np.random.RandomState(RNG_SEED)
    train_mask = np.zeros(len(metadata), dtype=bool)
    for sess, idx in metadata.groupby(session_col).groups.items():
        idx = np.array(idx)
        rng.shuffle(idx)
        n_train = int(0.7 * len(idx))
        train_mask[idx[:n_train]] = True
    return train_mask, ~train_mask


def cross_session_split(metadata, train_session, test_session, session_col="session"):
    train_mask = (metadata[session_col] == train_session).values
    test_mask = (metadata[session_col] == test_session).values
    return train_mask, test_mask


def cross_subject_splits(metadata, subject_col="subject"):
    """Leave-one-subject-out generator over unique subjects."""
    subjects = metadata[subject_col].unique()
    for held_out in subjects:
        test_mask = (metadata[subject_col] == held_out).values
        train_mask = ~test_mask
        yield held_out, train_mask, test_mask


## 9. Run: baselines across all three protocols, one subject at a time

In [ ]:
def score(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro")}


all_results = []
per_subject_data = {}

for subj in subjects:
    X, y, metadata = load_subject_epochs(subj)
    per_subject_data[subj] = (X, y, metadata)
    sfreq = dataset.interval and 250 or 250  # BNCI2014_001 native rate

    # --- within-session ---
    tr_mask, te_mask = within_session_split(metadata)
    for name, pipe in [("csp_lda", csp_lda_pipeline())]:
        pipe.fit(X[tr_mask], y[tr_mask])
        pred = pipe.predict(X[te_mask])
        all_results.append({"subject": subj, "protocol": "within_session", "model": name,
                             **score(y[te_mask], pred)})

    fbcsp = FBCSPClassifier(sfreq=sfreq)
    fbcsp.fit(X[tr_mask], y[tr_mask])
    pred = fbcsp.predict(X[te_mask])
    all_results.append({"subject": subj, "protocol": "within_session", "model": "fbcsp",
                         **score(y[te_mask], pred)})

    if HAS_PYRIEMANN:
        riem = riemannian_pipeline()
        riem.fit(X[tr_mask], y[tr_mask])
        pred = riem.predict(X[te_mask])
        all_results.append({"subject": subj, "protocol": "within_session", "model": "riemannian_ts_logreg",
                             **score(y[te_mask], pred)})

    # --- cross-session ---
    sessions = metadata["session"].unique()
    if len(sessions) >= 2:
        tr_mask, te_mask = cross_session_split(metadata, sessions[0], sessions[1])
        riem = riemannian_pipeline() if HAS_PYRIEMANN else csp_lda_pipeline()
        riem.fit(X[tr_mask], y[tr_mask])
        pred = riem.predict(X[te_mask])
        model_name = "riemannian_ts_logreg" if HAS_PYRIEMANN else "csp_lda"
        all_results.append({"subject": subj, "protocol": "cross_session", "model": model_name,
                             **score(y[te_mask], pred)})

results_within_cross = pd.DataFrame(all_results)
display(results_within_cross)


## 10. Cross-subject (LOSO) with the best baseline

In [ ]:
X_all = np.concatenate([per_subject_data[s][0] for s in subjects])
y_all = np.concatenate([per_subject_data[s][1] for s in subjects])
meta_all = pd.concat(
    [per_subject_data[s][2].assign(subject=s) for s in subjects], ignore_index=True
)

loso_results = []
for held_out, tr_mask, te_mask in cross_subject_splits(meta_all):
    pipe = riemannian_pipeline() if HAS_PYRIEMANN else csp_lda_pipeline()
    pipe.fit(X_all[tr_mask], y_all[tr_mask])
    pred = pipe.predict(X_all[te_mask])
    model_name = "riemannian_ts_logreg" if HAS_PYRIEMANN else "csp_lda"
    loso_results.append({"subject": held_out, "protocol": "cross_subject", "model": model_name,
                          **score(y_all[te_mask], pred)})

loso_df = pd.DataFrame(loso_results)
display(loso_df)
print(f"Chance level for 4 classes: 25%. Mean LOSO accuracy: {loso_df['accuracy'].mean():.1%}")


## 11. Deep models (EEGNet, ShallowFBCSPNet) — within-session

In [ ]:
if HAS_BRAINDECODE:
    deep_results = []
    for subj in subjects:
        X, y, metadata = per_subject_data[subj]
        tr_mask, te_mask = within_session_split(metadata)
        n_channels, n_times = X.shape[1], X.shape[2]
        n_classes = len(np.unique(y))

        y_idx = pd.Categorical(y).codes  # braindecode wants 0..n_classes-1 ints
        Xf = X.astype(np.float32)

        for model_name, builder in [("eegnet", make_eegnet), ("shallow_fbcsp", make_shallow_fbcsp)]:
            module = builder(n_channels, n_classes, n_times)
            clf = make_eeg_classifier(module, max_epochs=50)
            clf.fit(Xf[tr_mask], y_idx[tr_mask])
            pred = clf.predict(Xf[te_mask])
            deep_results.append({"subject": subj, "protocol": "within_session", "model": model_name,
                                  **score(y_idx[te_mask], pred)})
    deep_df = pd.DataFrame(deep_results)
    display(deep_df)
else:
    print("Install braindecode + skorch to run the deep-model arms.")


## 12. The differentiator: rest class + false activations per minute

Add a rest/no-control class (inter-trial or eyes-open resting segments), then sweep a confidence threshold and report **false activations/min**, not just accuracy — a spurious command is a safety event for any device that moves.

In [ ]:
def add_rest_epochs(X, y, metadata, raw_rest_segments=None, n_synthetic=None):
    """If the dataset exposes explicit resting-state segments, epoch those as the
    rest class. MOABB's MotorImagery paradigm doesn't expose rest by default — as a
    documented fallback, synthesize a 'rest' class from inter-trial baseline windows
    if you don't have real resting recordings. Prefer real rest data when available."""
    if raw_rest_segments is not None:
        X_rest = raw_rest_segments
    else:
        n_synthetic = n_synthetic or len(X) // 4
        rng = np.random.RandomState(RNG_SEED)
        idx = rng.choice(len(X), size=n_synthetic, replace=False)
        # crude stand-in: shuffle time within existing epochs to destroy task structure
        X_rest = X[idx].copy()
        for i in range(len(X_rest)):
            perm = rng.permutation(X_rest.shape[-1])
            X_rest[i] = X_rest[i][:, perm]
    y_rest = np.full(len(X_rest), CFG.resting_label, dtype=object)
    X_aug = np.concatenate([X, X_rest])
    y_aug = np.concatenate([y.astype(object), y_rest])
    return X_aug, y_aug


def false_activations_per_min(y_true, probs, classes, rest_label, thresholds, epoch_duration_s):
    """For each confidence threshold, count how often a non-rest class is emitted
    above threshold when the true label is rest — normalize to a per-minute rate."""
    rest_idx = list(classes).index(rest_label)
    rows = []
    is_rest = (y_true == rest_label)
    n_rest_epochs = is_rest.sum()
    total_rest_minutes = n_rest_epochs * epoch_duration_s / 60.0
    for th in thresholds:
        pred_class = probs.argmax(axis=1)
        pred_conf = probs.max(axis=1)
        false_activation = is_rest & (pred_class != rest_idx) & (pred_conf >= th)
        rate = false_activation.sum() / max(total_rest_minutes, 1e-8)
        rows.append({"threshold": th, "false_activations_per_min": rate})
    return pd.DataFrame(rows)


if HAS_PYRIEMANN:
    subj = subjects[0]
    X, y, metadata = per_subject_data[subj]
    X_aug, y_aug = add_rest_epochs(X, y, metadata)
    tr_mask = np.zeros(len(y_aug), dtype=bool)
    rng = np.random.RandomState(RNG_SEED)
    idx = rng.permutation(len(y_aug))
    tr_mask[idx[: int(0.7 * len(idx))]] = True

    riem = Pipeline([
        ("cov", Covariances(estimator="oas")),
        ("ts", TangentSpace(metric="riemann")),
        ("logreg", LogisticRegression(max_iter=1000)),
    ])
    riem.fit(X_aug[tr_mask], y_aug[tr_mask])
    probs = riem.predict_proba(X_aug[~tr_mask])
    classes = riem.named_steps["logreg"].classes_

    fa_df = false_activations_per_min(
        y_aug[~tr_mask], probs, classes, CFG.resting_label,
        thresholds=np.linspace(0.25, 0.95, 15), epoch_duration_s=(CFG.tmax - CFG.tmin),
    )
    display(fa_df)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(fa_df["threshold"], fa_df["false_activations_per_min"], "o-")
    ax.set_xlabel("confidence threshold")
    ax.set_ylabel("false activations / min")
    ax.set_title(f"Subject {subj}: false-activation rate vs. confidence threshold")
    plt.tight_layout()
    plt.savefig("project2_false_activations.png", dpi=150)
    plt.show()


## Notes / expected numbers

- **Within-subject**: expect 70–80% accuracy; **cross-subject (LOSO)**: expect 50–65%. Chance for 4 classes is 25% — state it explicitly next to every number.
- The Riemannian tangent-space pipeline is usually the best accuracy-per-FLOP baseline here — cheaper than FBCSP and often competitive with or ahead of the deep models on a dataset this small (9 subjects).
- `add_rest_epochs`'s synthetic fallback (time-shuffling existing epochs) is a placeholder — swap in real resting-state recordings if your acquisition protocol captures them; synthetic rest will understate real false-activation risk because shuffled-epoch 'rest' is spectrally unrealistic.
- This notebook's fitted Riemannian/CSP pipelines and epoch-loading code (`load_subject_epochs`, `per_subject_data`) are reused directly in [Project 3](02_eeg_emg_fusion.ipynb).
